In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("jobs_cleaned_missing_treated.csv")

In [3]:
print(df.shape)
print(df.columns.tolist())

(7269, 42)
['absolute_url', 'ai_opt_out_request_url', 'category', 'city', 'company_name', 'content_clean', 'contract_time', 'contract_type', 'created', 'created_date', 'created_time', 'demographic_data_consent_applies', 'departments_clean', 'education', 'ic_or_mg', 'id', 'include_ai_disclaimer', 'internal_job_id', 'job_level', 'job_type', 'language_clean', 'latitude', 'location_clean', 'longitude', 'offices_clean', 'other_market_salary_currency', 'other_market_salary_max', 'other_market_salary_min', 'pay_transparency_max', 'pay_transparency_min', 'pay_transparency_unit', 'region', 'requisition_id', 'salary_currency', 'salary_max', 'salary_min', 'standard_role', 'state', 'title', 'updated_at', 'working_model_clean', 'source']


In [4]:
# Identifier / metadata columns
identifier_cols = ['id', 'internal_job_id','requisition_id','absolute_url','ai_opt_out_request_url']

In [5]:
# Text columns
text_cols = ['title','content_clean','company_name', 'location_clean', 'offices_clean', 'departments_clean']

In [6]:
# Date / time columns
datetime_cols = ['created','created_date','created_time','updated_at']

In [7]:
# Target / salary-related columns
salary_cols = ['salary_min','salary_max','salary_currency','other_market_salary_min','other_market_salary_max','other_market_salary_currency',
    'pay_transparency_min','pay_transparency_max','pay_transparency_unit']

In [8]:
# Categorical / structured feature columns
categorical_cols = [ 'category','city','state','standard_role','contract_time','contract_type',
    'education','ic_or_mg','job_level','job_type','language_clean', 'region','working_model_clean','source']

In [9]:
# Numerical feature columns
numeric_cols = ['latitude', 'longitude']

In [10]:
print("Identifier:", len(identifier_cols))
print("Text:", len(text_cols))
print("Datetime:", len(datetime_cols))
print("Salary:", len(salary_cols))
print("Categorical:", len(categorical_cols))
print("Numerical:", len(numeric_cols))

Identifier: 5
Text: 6
Datetime: 4
Salary: 9
Categorical: 14
Numerical: 2


In [11]:
df.dtypes

absolute_url                         object
ai_opt_out_request_url               object
category                             object
city                                 object
company_name                         object
content_clean                        object
contract_time                        object
contract_type                        object
created                              object
created_date                         object
created_time                         object
demographic_data_consent_applies    float64
departments_clean                    object
education                            object
ic_or_mg                             object
id                                    int64
include_ai_disclaimer                object
internal_job_id                     float64
job_level                            object
job_type                             object
language_clean                       object
latitude                            float64
location_clean                  

In [12]:
df['created'] = pd.to_datetime(df['created'], errors='coerce', utc=True)

In [13]:
df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')

In [14]:
df['created_time'] = pd.to_datetime(df['created_time'],format='%H:%M:%S', errors='coerce')

In [15]:
df['updated_at'] = pd.to_datetime(df['updated_at'], errors='coerce', utc=True)

In [16]:
print(df[['created', 'created_date', 'created_time', 'updated_at']].dtypes)

created         datetime64[ns, UTC]
created_date         datetime64[ns]
created_time         datetime64[ns]
updated_at      datetime64[ns, UTC]
dtype: object


In [17]:
df['posting_year'] = df['created_date'].dt.year

In [18]:
df['posting_month'] = df['created_date'].dt.month

In [19]:
df['posting_day'] = df['created_date'].dt.day

In [20]:
df['posting_dayofweek'] = df['created_date'].dt.dayofweek

In [21]:
df['posting_weekend'] = df['posting_dayofweek'].isin([5, 6]).astype(int)

In [22]:
print(df[['created_date','posting_year','posting_month','posting_day',
          'posting_dayofweek','posting_weekend']].head())

  created_date  posting_year  posting_month  posting_day  posting_dayofweek  \
0   2026-07-03          2026              7            3                  4   
1   2026-07-10          2026              7           10                  4   
2   2026-07-03          2026              7            3                  4   
3   2026-07-09          2026              7            9                  3   
4   2026-07-03          2026              7            3                  4   

   posting_weekend  
0                0  
1                0  
2                0  
3                0  
4                0  


In [23]:
df[['salary_min','salary_max','other_market_salary_min','other_market_salary_max',
    'pay_transparency_min','pay_transparency_max']].describe()

,salary_min,salary_max,other_market_salary_min,other_market_salary_max,pay_transparency_min,pay_transparency_max
count,2.616000e+03,2.915000e+03,50.000000,50.000000,196.000000,196.000000
mean,6.406804e+05,1.315851e+06,124960.464000,124960.464000,151227.040816,199045.918367
std,5.693474e+05,1.055261e+06,34280.630294,34280.630294,59780.931128,75934.809650
min,3.000000e+03,2.000000e+04,50003.200000,50003.200000,49000.000000,50000.000000
25%,2.080000e+05,5.000000e+05,104000.000000,104000.000000,110000.000000,133750.000000
50%,5.000000e+05,1.200000e+06,134000.000000,134000.000000,146000.000000,195000.000000
75%,9.000000e+05,1.800000e+06,148875.000000,148875.000000,192000.000000,240000.000000
max,4.500000e+06,9.000000e+06,216000.000000,216000.000000,330000.000000,484000.000000


In [24]:
df['has_salary'] = ( df['salary_min'].notna() & df['salary_max'].notna()).astype(int)
df['has_salary'].value_counts()

has_salary
0    4653
1    2616
Name: count, dtype: int64

In [25]:
df['salary_range'] = (df['salary_max'] - df['salary_min'])
print(df[['salary_min', 'salary_max', 'salary_range']].head(10))

   salary_min  salary_max  salary_range
0         NaN         NaN           NaN
1         NaN         NaN           NaN
2         NaN         NaN           NaN
3         NaN         NaN           NaN
4         NaN         NaN           NaN
5         NaN         NaN           NaN
6         NaN         NaN           NaN
7         NaN         NaN           NaN
8         NaN         NaN           NaN
9         NaN         NaN           NaN


In [26]:
df.loc[ df['salary_range'].notna(),['salary_min', 'salary_max', 'salary_range']].head(10)

,salary_min,salary_max,salary_range
184,1200000.0,1999992.0,799992.0
185,1800000.0,2100000.0,300000.0
187,1200000.0,1699992.0,499992.0
188,960000.0,999996.0,39996.0
189,1320000.0,1699992.0,379992.0
200,700000.0,1400000.0,700000.0
209,100000.0,1500000.0,1400000.0
210,1400000.0,1500000.0,100000.0
211,500000.0,1500000.0,1000000.0
212,800000.0,2500000.0,1700000.0


In [27]:
df['salary_mid'] = ((df['salary_min'] + df['salary_max']) / 2)

print(df.loc[df['salary_mid'].notna(), ['salary_min', 'salary_max', 'salary_mid']].head(10))

     salary_min  salary_max  salary_mid
184   1200000.0   1999992.0   1599996.0
185   1800000.0   2100000.0   1950000.0
187   1200000.0   1699992.0   1449996.0
188    960000.0    999996.0    979998.0
189   1320000.0   1699992.0   1509996.0
200    700000.0   1400000.0   1050000.0
209    100000.0   1500000.0    800000.0
210   1400000.0   1500000.0   1450000.0
211    500000.0   1500000.0   1000000.0
212    800000.0   2500000.0   1650000.0


In [28]:
df[['has_salary', 'salary_range', 'salary_mid']].isna().sum()

has_salary         0
salary_range    4653
salary_mid      4653
dtype: int64

In [29]:
df['description_length'] = df['content_clean'].str.len()
print(df[['content_clean', 'description_length']].head())

                                       content_clean  description_length
0  This job is with ABB, an inclusive employer an...                 500
1  This job is with Kyndryl, an inclusive employe...                 500
2  This job is with Cornerstone OnDemand, an incl...                 500
3  This job is with Morningstar, an inclusive emp...                 500
4  This job is with Morningstar, an inclusive emp...                 500


In [30]:
df['description_length'].value_counts().head(10)

description_length
500    4014
499     101
498      84
494      60
488      59
490      56
496      54
492      48
486      33
497      18
Name: count, dtype: int64

In [31]:
df.drop(columns=['description_length'], inplace=True)

In [32]:
df['company_job_count'] = df.groupby('company_name')['company_name'].transform('count')

In [33]:
df[['company_name', 'company_job_count']] .head(10)

,company_name,company_job_count
0,ABB,25
1,Kyndryl,71
2,Cornerstone OnDemand,23
3,Morningstar,32
4,Morningstar,32
5,Cornerstone OnDemand,23
6,eBay,45
7,Morningstar,32
8,ABB,25
9,ABB,25


In [34]:
df['company_name'].value_counts().head(10)

company_name
Stripe         524
Datadog        421
MongoDB        381
Brex           266
Figma          169
Flexport       148
Asana          146
Accenture      116
Gusto, Inc.     78
Kyndryl         71
Name: count, dtype: int64

In [36]:
df['created_date'].min()

Timestamp('2019-04-17 00:00:00')

In [37]:
df['created_date'].max()

Timestamp('2026-07-14 00:00:00')

In [38]:
latest_day=df['created_date'].max()
df['posting_age_days']=(latest_day-df['created_date']).dt.days

In [39]:
print(df[['created_date', 'posting_age_days']].head(10))

  created_date  posting_age_days
0   2026-07-03                11
1   2026-07-10                 4
2   2026-07-03                11
3   2026-07-09                 5
4   2026-07-03                11
5   2026-07-03                11
6   2026-07-03                11
7   2026-07-03                11
8   2026-07-03                11
9   2026-07-03                11


In [40]:
df['posting_quarter'] = df['created_date'].dt.quarter

In [41]:
 df[['created_date', 'posting_quarter']].head(10)

,created_date,posting_quarter
0,2026-07-03,3
1,2026-07-10,3
2,2026-07-03,3
3,2026-07-09,3
4,2026-07-03,3
5,2026-07-03,3
6,2026-07-03,3
7,2026-07-03,3
8,2026-07-03,3
9,2026-07-03,3


In [42]:
df['has_salary_currency']=(df['salary_currency'].notna()).astype(int)

In [43]:
df['has_salary_currency'].value_counts()

has_salary_currency
0    6667
1     602
Name: count, dtype: int64

In [45]:
df['has_other_market_salary'] = (df['other_market_salary_min'].notna() & df['other_market_salary_max'].notna()).astype(int)

In [46]:
df['has_other_market_salary'].value_counts()

has_other_market_salary
0    7219
1      50
Name: count, dtype: int64

In [47]:
df['has_pay_transparency'] = (df['pay_transparency_min'].notna() & df['pay_transparency_max'].notna()).astype(int)

In [48]:
df['has_pay_transparency'].value_counts()

has_pay_transparency
0    7073
1     196
Name: count, dtype: int64

In [49]:
df['has_city'] = ( df['city'].notna()).astype(int)


In [50]:
df['has_city'].value_counts()

has_city
1    3816
0    3453
Name: count, dtype: int64

In [51]:
df.drop(columns=['has_city'], inplace=True)

In [52]:
df['salary_range_ratio'] = df['salary_max'] / df['salary_min']
print(df['salary_range_ratio'].describe())

count    2616.000000
mean        2.501561
std         3.199906
min         1.000000
25%         1.428571
50%         1.800000
75%         2.500000
max        83.333333
Name: salary_range_ratio, dtype: float64


In [53]:
df['title_word_count'] = df['title'].str.split().str.len()

In [54]:
print(df['title_word_count'].describe())

count    7269.000000
mean        3.405145
std         1.804181
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        18.000000
Name: title_word_count, dtype: float64


In [55]:
df['title_length'] = df['title'].str.len()

In [56]:
print(df['title_length'].describe())

count    7269.000000
mean       25.973999
std        13.374265
min         3.000000
25%        16.000000
50%        20.000000
75%        33.000000
max       124.000000
Name: title_length, dtype: float64


In [57]:
df['salary_range_pct'] = ((df['salary_max'] - df['salary_min']) / df['salary_min']) * 100

In [58]:
df['salary_range_pct'].describe()

count    2616.000000
mean      150.156139
std       319.990575
min         0.000000
25%        42.857143
50%        80.000000
75%       150.000000
max      8233.333333
Name: salary_range_pct, dtype: float64

In [59]:
df.drop(columns=['salary_range_ratio'], inplace=True)

In [62]:
df['job_level'].value_counts()

job_level
Professional - IC    202
IC - Professional    123
People Manager        26
Senior Manager        11
Director               9
Manager                5
Senior Director        3
Support                1
Vice President         1
Name: count, dtype: int64

In [63]:
df['job_level_clean'] = df['job_level'].replace({ 'IC - Professional': 'Professional - IC'})

In [64]:
df['job_level_clean'].value_counts()

job_level_clean
Professional - IC    325
People Manager        26
Senior Manager        11
Director               9
Manager                5
Senior Director        3
Support                1
Vice President         1
Name: count, dtype: int64

In [65]:
def seniority_group(x):
    if pd.isna(x):
        return pd.NA
    elif x == 'Professional - IC':
        return 'Individual Contributor'
    elif x in ['Manager', 'People Manager']:
        return 'Manager'
    elif x == 'Senior Manager':
        return 'Senior Manager'
    elif x in ['Director', 'Senior Director']:
        return 'Director'
    elif x == 'Vice President':
        return 'Executive'
    else:
        return 'Other'

In [66]:
df['seniority_group'] = df['job_level_clean'].apply(seniority_group)

In [67]:
df['seniority_group'].value_counts(dropna=False)

seniority_group
<NA>                      6888
Individual Contributor     325
Manager                     31
Director                    12
Senior Manager              11
Other                        1
Executive                    1
Name: count, dtype: int64

In [68]:
df['has_job_level'] = df['job_level_clean'].notna().astype(int)

In [69]:
df['has_job_level'].value_counts()

has_job_level
0    6888
1     381
Name: count, dtype: int64

In [70]:
df['has_work_model'] = df['working_model_clean'].notna().astype(int)

In [71]:
df['has_work_model'].value_counts()

has_work_model
0    6888
1     381
Name: count, dtype: int64

In [72]:
df['has_education'] = df['education'].notna().astype(int)

In [73]:
print(df['has_education'].value_counts())

has_education
0    7109
1     160
Name: count, dtype: int64


In [75]:
df['has_location'] = df['location_clean'].notna().astype(int)
df['has_location'].value_counts()

has_location
1    7261
0       8
Name: count, dtype: int64

In [76]:
df.drop(columns=['has_location'], inplace=True)

In [77]:
df['title_has_senior'] = ( df['title'].str.contains('senior', case=False, na=False)).astype(int)

print(df['title_has_senior'].value_counts())

title_has_senior
0    6490
1     779
Name: count, dtype: int64


In [78]:
df['title_has_lead'] = ( df['title'].str.contains('lead', case=False, na=False)).astype(int)
print(df['title_has_lead'].value_counts())

title_has_lead
0    7058
1     211
Name: count, dtype: int64


In [79]:
df['title_has_manager'] = (df['title'].str.contains('manager', case=False, na=False)).astype(int)
print(df['title_has_manager'].value_counts())

title_has_manager
0    6376
1     893
Name: count, dtype: int64


In [80]:
df['title_has_director'] = ( df['title'].str.contains('director', case=False, na=False)).astype(int)
print(df['title_has_director'].value_counts())

title_has_director
0    7184
1      85
Name: count, dtype: int64


In [81]:
df['title_has_intern'] = (df['title'].str.contains('intern', case=False, na=False)).astype(int)
print(df['title_has_intern'].value_counts())

title_has_intern
0    7232
1      37
Name: count, dtype: int64


In [82]:
df['title_has_junior'] = ( df['title'].str.contains('junior', case=False, na=False)).astype(int)
print(df['title_has_junior'].value_counts())

title_has_junior
0    7262
1       7
Name: count, dtype: int64


In [83]:
df['title_has_principal'] = (df['title'].str.contains('principal', case=False, na=False)).astype(int)
print(df['title_has_principal'].value_counts())

title_has_principal
0    7198
1      71
Name: count, dtype: int64


In [84]:
df['title_has_staff'] = (df['title'].str.contains('staff', case=False, na=False)).astype(int)
print(df['title_has_staff'].value_counts())

title_has_staff
0    7038
1     231
Name: count, dtype: int64


In [85]:
df['title_has_associate'] = (df['title'].str.contains('associate', case=False, na=False)).astype(int)
print(df['title_has_associate'].value_counts())

title_has_associate
0    7135
1     134
Name: count, dtype: int64


In [86]:
feature_cols = [
    'posting_year',
    'posting_month',
    'posting_day',
    'posting_dayofweek',
    'posting_weekend',
    'posting_age_days',
    'posting_quarter',
    'has_salary',
    'salary_range',
    'salary_mid',
    'has_salary_currency',
    'has_other_market_salary',
    'has_pay_transparency',
    'company_job_count',
    'title_word_count',
    'title_length',
    'salary_range_pct',
    'job_level_clean',
    'seniority_group',
    'has_job_level',
    'has_work_model',
    'has_education',
    'title_has_senior',
    'title_has_lead',
    'title_has_manager',
    'title_has_director',
    'title_has_intern',
    'title_has_junior',
    'title_has_principal',
    'title_has_staff',
    'title_has_associate'
]

print(df[feature_cols].dtypes)

posting_year                 int32
posting_month                int32
posting_day                  int32
posting_dayofweek            int32
posting_weekend              int32
posting_age_days             int64
posting_quarter              int32
has_salary                   int32
salary_range               float64
salary_mid                 float64
has_salary_currency          int32
has_other_market_salary      int32
has_pay_transparency         int32
company_job_count            int64
title_word_count             int64
title_length                 int64
salary_range_pct           float64
job_level_clean             object
seniority_group             object
has_job_level                int32
has_work_model               int32
has_education                int32
title_has_senior             int32
title_has_lead               int32
title_has_manager            int32
title_has_director           int32
title_has_intern             int32
title_has_junior             int32
title_has_principal 

In [87]:
def salary_availability(row):
    if pd.notna(row['salary_min']) and pd.notna(row['salary_max']):
        return 'Both'
    elif pd.isna(row['salary_min']) and pd.notna(row['salary_max']):
        return 'Max only'
    elif pd.notna(row['salary_min']) and pd.isna(row['salary_max']):
        return 'Min only'
    else:
        return 'No salary'

In [88]:
df['salary_availability'] = df.apply(salary_availability, axis=1)

In [89]:
print(df['salary_availability'].value_counts())

salary_availability
No salary    4354
Both         2616
Max only      299
Name: count, dtype: int64


In [90]:
print("Negative salary range:", (df['salary_range'] < 0).sum())
print("Negative salary midpoint:", (df['salary_mid'] < 0).sum())
print("Negative salary range %:", (df['salary_range_pct'] < 0).sum())

Negative salary range: 0
Negative salary midpoint: 0
Negative salary range %: 0


In [91]:
invalid_salary = df[df['salary_min'].notna() & df['salary_max'].notna() &(df['salary_max'] < df['salary_min'])]

In [92]:
print("Invalid salary records:", len(invalid_salary))

Invalid salary records: 0


In [93]:
salary_without_currency = df[(df['salary_min'].notna() | df['salary_max'].notna()) &(df['salary_currency'].isna())]

print("Salary records without currency:", len(salary_without_currency))

Salary records without currency: 2391


In [95]:
def other_market_salary_availability(row):
    if pd.notna(row['other_market_salary_min']) and pd.notna(row['other_market_salary_max']):
        return 'Both'
    elif pd.isna(row['other_market_salary_min']) and pd.notna(row['other_market_salary_max']):
        return 'Max only'
    elif pd.notna(row['other_market_salary_min']) and pd.isna(row['other_market_salary_max']):
        return 'Min only'
    else:
        return 'No salary'

In [96]:
df['other_market_salary_availability'] = df.apply( other_market_salary_availability,axis=1)

In [97]:
print(df['other_market_salary_availability'].value_counts())

other_market_salary_availability
No salary    7219
Both           50
Name: count, dtype: int64


In [98]:
df.drop(columns=['other_market_salary_availability'], inplace=True)

In [100]:
print(df[['pay_transparency_min', 'pay_transparency_max']].notna() .sum())


pay_transparency_min    196
pay_transparency_max    196
dtype: int64


In [101]:
invalid_other_market = df[df['other_market_salary_min'].notna() &df['other_market_salary_max'].notna() &
df['other_market_salary_max'] < df['other_market_salary_min'])]
print("Invalid other-market salary records:", len(invalid_other_market))

Invalid other-market salary records: 0


In [102]:
invalid_pay_transparency = df[df['pay_transparency_min'].notna() & df['pay_transparency_max'].notna() &
    (df['pay_transparency_max'] < df['pay_transparency_min'])]
print("Invalid pay transparency records:", len(invalid_pay_transparency))

Invalid pay transparency records: 0


In [103]:
df.columns.duplicated().sum()

0

In [105]:
df.duplicated().sum()

0

In [106]:
print("Shape:", df.shape)

Shape: (7269, 74)


In [108]:
for i, col in enumerate(df.columns, start=1):
    print(i, col)

1 absolute_url
2 ai_opt_out_request_url
3 category
4 city
5 company_name
6 content_clean
7 contract_time
8 contract_type
9 created
10 created_date
11 created_time
12 demographic_data_consent_applies
13 departments_clean
14 education
15 ic_or_mg
16 id
17 include_ai_disclaimer
18 internal_job_id
19 job_level
20 job_type
21 language_clean
22 latitude
23 location_clean
24 longitude
25 offices_clean
26 other_market_salary_currency
27 other_market_salary_max
28 other_market_salary_min
29 pay_transparency_max
30 pay_transparency_min
31 pay_transparency_unit
32 region
33 requisition_id
34 salary_currency
35 salary_max
36 salary_min
37 standard_role
38 state
39 title
40 updated_at
41 working_model_clean
42 source
43 posting_year
44 posting_month
45 posting_day
46 posting_dayofweek
47 posting_weekend
48 has_salary
49 salary_range
50 salary_mid
51 company_job_count
52 posting_age_days
53 posting_quarter
54 has_salary_currency
55 has_other_market_salary
56 has_pay_transparency
57 title_word_count


In [109]:
constant_cols = [col for col in df.columns
    if df[col].nunique(dropna=False) <= 1]

print("Constant columns:")
print(constant_cols)

Constant columns:
[]


In [110]:
df.to_csv("jobs_feature_engineered.csv", index=False)